# 05 · Findings summary

Consolidated results of the virtual-site export validation suite (notebooks
01–04). This notebook reads the `results_*.json` written by the earlier
notebooks and renders a single pass/fail table, notes what "independent"
validation means for these pathways, records the history and verification of
interchange #1498, and states the deferred hybrid-topology follow-up.

In [1]:
import glob
import json

import pandas as pd

with open("provenance.json") as fh:
    prov = json.load(fh)
print("Certified against:")
for pkg in ["openff-toolkit", "openff-interchange", "openmm", "openmmforcefields",
            "openforcefields", "openff-nagl-models"]:
    print(f"  {pkg:22s} {prov['versions'].get(pkg)}")

rows = []
for path in sorted(glob.glob("results_0[1-4].json")):
    with open(path) as fh:
        rows.extend(json.load(fh))
df = pd.DataFrame(rows)
print(f"\nLoaded {len(df)} checks from {sorted(glob.glob('results_0[1-4].json'))}")

Certified against:
  openff-toolkit         0.18.1
  openff-interchange     0.5.4.dev4+g7e742e906
  openmm                 8.5.2
  openmmforcefields      0.16.0
  openforcefields        2026.1.0
  openff-nagl-models     2025.9.0

Loaded 65 checks from ['results_01.json', 'results_02.json', 'results_03.json', 'results_04.json']


## Pass/fail overview

Every check is `PASS`. The suite runs on an Interchange build (`main`, `> 0.5.3`)
that includes the fix for #1498, and notebook 01 *verifies* that fix rather than
recording it as an outstanding bug.

In [2]:
status_counts = df["status"].value_counts()
print("Status counts:")
for s, c in status_counts.items():
    print(f"  {s:20s} {c}")

# per-notebook rollup
print("\nBy notebook:")
for nb, sub in df.groupby("notebook"):
    statuses = sub["status"].value_counts().to_dict()
    print(f"  notebook {nb}: {len(sub)} checks -> {statuses}")

Status counts:
  PASS                 65

By notebook:
  notebook 01: 30 checks -> {'PASS': 30}
  notebook 02: 9 checks -> {'PASS': 9}
  notebook 03: 21 checks -> {'PASS': 21}
  notebook 04: 5 checks -> {'PASS': 5}


## What "independent" means here

`openmmforcefields`' `SMIRNOFFTemplateGenerator` builds its OpenMM `System` by
calling `ForceField.create_openmm_system`, which **is Interchange** — it then
serializes to an `ffxml` and lets `openmm.app.ForceField` rebuild it. So the two
routes in notebook 01 are *not* independent engines: they share Interchange's
virtual-site math, and a bug inside it would cancel in that comparison. Notebook
01 rigorously validates the `openmmforcefields`-specific step (the `ffxml`
round-trip — parameters, charges, and the full exception *table* — plus the
split-force and PME layouts). The checks that pin Interchange's virtual-site
math against a **truly independent** reference are:

- **notebook 02** — native OpenMM/AMBER water models (`OutOfPlaneSite` /
`ThreeParticleAverageSite`), a different code path entirely;
- **notebook 03** — closed-form geometry, **bond-graph-derived exclusion sets**,
and **hand-summed Coulomb + Lennard-Jones energies** (including scaled vsite
1-4 pairs at non-default scaling), which depend on nothing but the graph and
the textbook nonbonded expressions.

## Feature coverage matrix

| Theme | What was validated | Result |
|-------|--------------------|--------|
| **A** Interchange ↔ `openmmforcefields` round-trip | 4 vsite types × 4 charge methods (OFFXML `LibraryCharges`/`ToolkitAM1BCC`/`ChargeIncrementModel` + preset NAGL); net-charged molecule; reordering & intra-molecule remap; duplicate/mixed topologies; full exception-table equality; non-default *distinct* 1-4 scaling; split-force layout (NoCutoff **and** PME) | parameters/exceptions/energies/forces identical to ~1e-8; **#1498** fix verified |
| **B** OpenFF vs external water | TIP5P / TIP4P-Ew / OPC: `LocalCoordinatesSite` vs `OutOfPlane`/`ThreeParticleAverage` placement, charges, LJ, exclusions, PME box energy; random-orientation equivariance | vsite placement to ~1e-8 Å; PME energy to ~1e-7 |
| **C** Hand-calc / first-principles | closed-form geometry for all 4 types incl. general out-of-plane MonovalentLonePair (**#939**) and DivalentLonePair (**#1438**); charge-increment bookkeeping; 1-4 scaling incl. non-default (**#1436**); **exclusion set vs bond graph** (single & multi-molecule); **end-to-end hand-summed Coulomb+LJ energy** | all exact to ~1e-9 |
| **D** Dynamics | OPC/TIP5P water boxes + vsites in solute *and* solvent: stability, temperature, density; IC vs OMMFFS box energy | stable & consistent |

## Robustness — no silent-danger input values

OpenMM's exception model is hardwired (1-2/1-3 excluded, 1-5 full, each vsite
sharing one atom's exclusions). Notebook 01 confirms every SMIRNOFF setting that
would violate it is **rejected, not silently mis-handled**: non-zero `scale12`,
non-zero `scale13`, or non-unity `scale15` (electrostatics or vdW) raise in the
**OpenFF Toolkit** at parse time; an `exclusion_policy` other than `"parents"`
raises in **Interchange** at export (and again in `openmmforcefields`). Only
`scale14` is freely variable, and it is applied consistently by both routes.

## interchange #1498 — history and verification

**The bug.** When a virtual-site system was exported with
`combine_nonbonded_forces=False` (electrostatics and vdW as *separate* OpenMM
forces), virtual-site nonbonded **exceptions were incorrectly zeroed**, changing
the energy (a no-vsite control was unaffected, isolating it to virtual sites — on
Interchange 0.5.3 this was a 6.14 kcal/mol error for a `BondCharge` system with
scaled 1-4 vsite exceptions). The **default** `combine_nonbonded_forces=True`
path used by essentially all consumers was always correct.

**Status.** The fix was merged after the 0.5.3 release. This suite runs on the
patched Interchange (`main`), and notebook 01 verifies the fix is **correct and
complete**: split- and combined-force total energies now agree for every vsite
type, with a no-vsite control confirming the split-force machinery itself is
sound. The version gate in notebook 01 will fail if run against an unpatched
Interchange (≤ 0.5.3).

In [3]:
known = df[df["status"].astype(str).str.contains("KNOWN_BUG")]
if len(known):
    print("Unexpected known-bug rows (should be none on a patched Interchange):")
    print(known[["notebook", "vsite_type", "max_energy_delta_kcal", "status"]].to_string(index=False))
else:
    print("No known-bug rows: interchange #1498 fix verified in this environment.")

No known-bug rows: interchange #1498 fix verified in this environment.


## Deferred follow-up: virtual sites through a hybrid-topology factory

This suite validates virtual-site **export** (SMIRNOFF → OpenMM `System`). A
separate downstream concern is copying `LocalCoordinatesSite` virtual sites into
a **hybrid-topology** alchemical `System` used for relative free-energy
calculations (tracked historically as the openfe #394/#395 hybrid-topology
virtual-site handling). Because Interchange now emits `LocalCoordinatesSite`
(rather than the `ThreeParticleAverage`/`OutOfPlane` types those factories were
originally written for), a follow-up notebook should:

1. build old/new end-state `System`s with `LocalCoordinatesSite` virtual sites,
2. construct the hybrid `System` via the factory,
3. confirm every virtual site survives with correct parent indices, exceptions,
and energies at both end states.

That requires the alchemical factory package as a dependency and is scoped as
the next iteration; the export foundations it depends on are validated here.

In [4]:
print("\n" + "=" * 70)
n_pass = int((df["status"] == "PASS").sum())
n_known = int(df["status"].astype(str).str.contains("KNOWN_BUG").sum())
n_other = len(df) - n_pass - n_known
print(f"SUMMARY: {n_pass} PASS, {n_known} KNOWN_BUG, {n_other} other")
assert n_known == 0, "Unexpected KNOWN_BUG rows on a patched Interchange!"
assert n_other == 0, "Unexpected non-PASS results present!"
print("Virtual-site export from Interchange to OpenMM (with and without "
      "openmmforcefields) is validated on the certified environment.")


SUMMARY: 65 PASS, 0 KNOWN_BUG, 0 other
Virtual-site export from Interchange to OpenMM (with and without openmmforcefields) is validated on the certified environment.
